# CA1

## Background

Grid search is a common tool in machine learning for tuning the hyperparameters of machine learning models to improve their predictive performance. However, because it relies on an exhaustive, brute-force search across a pre-defined grid of values, it is computationally expensive and does not guarantee that the optimal hyperparameters will be found. There are more advanced, efficient alternatives that can lead to better hyperparameters in less time. One such alternative is Optuna, a hyperparameter optimization framework. In this compulsory assignment, you will learn about and apply Optuna for the efficient optimisation of model hyperparameters.

## Context

You are a freshly graduated data scientist from university and you recently got hired in a junior data scientist position at Oslo University Hospital. Your department leader assigns you to a clinical research project where you are supposed to train machine learning models on clinical data. Extremely busy, as she usually is, your department leader hands you a handwritten note of what she wants you to do for the project before she runs off to the next meeting. 

 

**The note reads the following:**

- You got access to clinical data on liver measurements (hepatic_data.csv) and we need you to train a well performing logistic regression model (which will serve as a baseline model) on these data
- The target feature is column "Diagnosis"
- Use logistic regression from the scikit-learn package and optimise model performance using hyperparameters C and l1_ratio
- For hyperparameter optimisation use the Optuna package (https://optuna.org/). If you don't know how to use Optuna, get familiar with it and learn how to use it
- Carry out the hyperarameter search such that it uses the OptunaSearchCV. Make sure that it contains the following settings: 
    - a pipeline containing a StandardScaler and a LogisticRegression model
    - use  of repeated stratified KFold cross validation with 4 splits and 25 repeats
    - use of performance metric Matthews correlation coefficient (MCC) (scikit-learn) for optimisation
    - 200 trials for the optimisation process
    - set parameter "verbose" to 1
    - set parameter "n_jobs" to -1
- Measure and report the time it takes to complete the optimisation process

## Imports

In [1]:
import pandas as pd
import optuna
import time

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import matthews_corrcoef, make_scorer

from optuna.integration import OptunaSearchCV

c:\Users\milad\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\integration\sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)


## Loading & Checking Data

In [2]:
df = pd.read_csv("hepatic_data.csv")

# I am watching for outliers by comparing the median (50% quantile) and the min and max.
df.describe()

,Age,Gender,BMI,AlcoholConsumption,Smoking,GeneticRisk,PhysicalActivity,Diabetes,Hypertension,LiverFunctionTest,Diagnosis
count,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000
mean,50.394118,0.504118,27.699801,9.832309,0.291765,0.521765,5.000993,0.142353,0.154706,59.863867,0.550588
std,17.641915,0.500130,7.210400,5.757472,0.454708,0.666262,2.847074,0.349515,0.361730,22.996262,0.497581
min,20.000000,0.000000,15.004710,0.003731,0.000000,0.000000,0.001852,0.000000,0.000000,20.019254,0.000000
25%,35.000000,0.000000,21.455414,4.841811,0.000000,0.000000,2.622121,0.000000,0.000000,40.024216,0.000000
50%,51.000000,1.000000,27.925367,9.828195,0.000000,0.000000,5.022883,0.000000,0.000000,59.513146,1.000000
75%,66.000000,1.000000,33.957668,14.871671,1.000000,1.000000,7.401642,0.000000,0.000000,79.428755,1.000000
max,80.000000,1.000000,39.992845,19.952456,1.000000,2.000000,9.994964,1.000000,1.000000,99.991413,1.000000


Maybe there is a outlier in AlcoholConsumption since the median is 9.828~ and the max value is 19.9524~

In [3]:
# Getting information about the data type and the number of data with values in each columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1700 entries, 0 to 1699
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 1700 non-null   int64  
 1   Gender              1700 non-null   int64  
 2   BMI                 1700 non-null   float64
 3   AlcoholConsumption  1700 non-null   float64
 4   Smoking             1700 non-null   int64  
 5   GeneticRisk         1700 non-null   int64  
 6   PhysicalActivity    1700 non-null   float64
 7   Diabetes            1700 non-null   int64  
 8   Hypertension        1700 non-null   int64  
 9   LiverFunctionTest   1700 non-null   float64
 10  Diagnosis           1700 non-null   int64  
dtypes: float64(4), int64(7)
memory usage: 146.2 KB


In [4]:
# Here i am just checking the shape / how many features
df.shape

(1700, 11)

### Checking for missing data 

In [5]:
df.isnull().sum()

Age                   0
Gender                0
BMI                   0
AlcoholConsumption    0
Smoking               0
GeneticRisk           0
PhysicalActivity      0
Diabetes              0
Hypertension          0
LiverFunctionTest     0
Diagnosis             0
dtype: int64

no missing data

## Modelling

In [6]:
# Splitting the data to features and target
X = df.drop("Diagnosis", axis = 1)
y = df["Diagnosis"]

In [9]:
# Creating the pipeline with a scaler and a logistic regression with elasticnet penalty
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty = "elasticnet",
        solver = "saga",
        max_iter = 10000
    ))
])

# Creating the parameters for the hyperparameter
parameters = {
    "classifier__C": optuna.distributions.FloatDistribution(0.001, 100, log = True),
    "classifier__l1_ratio": optuna.distributions.FloatDistribution(0, 1)
}

In [10]:
# Creating the cross validation with 4 splits and 25 repeats
cv = RepeatedStratifiedKFold(
    n_splits=4,
    n_repeats=25,
    random_state=42
)

# Creating the scorer for the optimization
mcc_scorer = make_scorer(matthews_corrcoef)

# Creating the OptunaSearchCV object
optuna_search = OptunaSearchCV(
    estimator = pipeline,
    param_distributions = parameters,
    cv = cv,
    scoring = mcc_scorer,
    n_trials = 200,
    n_jobs = -1,
    verbose = 1,
    random_state = 42
)

C:\Users\milad\AppData\Local\Temp\ipykernel_16160\4117063543.py:12: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  optuna_search = OptunaSearchCV(


In [11]:
start_time = time.time()

optuna_search.fit(X, y)

end_time = time.time()

elapsed_time = end_time - start_time

print(f"Time: {elapsed_time:.2f} seconds")

[I 2026-09-04 14:36:35,950] A new study created in memory with name: no-name-c58b8cea-60e2-4c70-b013-073b4167b5b8
c:\Users\milad\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\terminator\erroreval.py:113: FutureWarning: `optuna.terminator` module has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  optuna_warn(_DEPRECATION_WARNING_MESSAGE, FutureWarning)
[I 2026-09-04 14:36:49,728] Trial 0 finished with value: 0.0 and parameters: {'classifier__C': 0.001819583515584144, 'classifier__l1_ratio': 0.7168572500932902}. Best is trial 0 with value: 0.0.
c:\Users\milad\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\terminator\erroreval.py:113: FutureWarning: `optuna.terminator` module has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  optuna_warn(_DEPRECATION_WARNING_MESSAGE, FutureWarning)
[I 2026-09-04

Time: 241.25 seconds


## Results

In [12]:
print("Best parameters:")
print(optuna_search.best_params_)

print("\nBest MCC score:")
print(optuna_search.best_score_)

print("\nBest trial number:")
print(optuna_search.best_trial_.number)

Best parameters:
{'classifier__C': 0.02435367356004619, 'classifier__l1_ratio': 0.036964625810111944}

Best MCC score:
0.654374495959908

Best trial number:
171
